## **Thanks for my partner** **Mst Rahi offer me the wonderful** queries to inprove my understanding in chapter 7

## **Query 1 – Long-Serving Employees (10+ years)**

**🟦 Proposition:**  
List all currently employed individuals who have worked for the company for more than 10 years. Helps HR identify loyal, long-serving employees.

**🟩 Why It’s Special:**  
Uses a combination of filtering (`EndDate IS NULL`) and `DATEDIFF` to isolate current employees with over 10 years of tenure. Clean and practical for service award planning.

In [32]:
USE AdventureWorks2017;
GO

SELECT 
    p.FirstName + ' ' + p.LastName AS EmployeeName,
    d.Name AS DepartmentName,
    edh.StartDate AS HireDate,
    DATEDIFF(YEAR, edh.StartDate, GETDATE()) AS YearsOfService
FROM HumanResources.EmployeeDepartmentHistory edh
JOIN Person.Person p ON edh.BusinessEntityID = p.BusinessEntityID
JOIN HumanResources.Department d ON edh.DepartmentID = d.DepartmentID
WHERE edh.EndDate IS NULL
  AND DATEDIFF(YEAR, edh.StartDate, GETDATE()) > 10
ORDER BY YearsOfService DESC;


Commands completed successfully.

(290 rows affected)

Total execution time: 00:00:00.067

EmployeeName,DepartmentName,HireDate,YearsOfService
Guy Gilbert,Production,2006-06-30,19
JoLynn Dobney,Production,2007-12-26,18
Roberto Tamburello,Engineering,2007-11-11,18
Thierry D'Hers,Tool Design,2007-12-11,18
Kevin Brown,Marketing,2007-01-26,18
Terri Duffy,Engineering,2008-01-31,17
Diane Margheim,Research and Development,2008-12-29,17
Gail Erickson,Engineering,2008-01-06,17
Jossef Goldberg,Engineering,2008-01-24,17
Sariya Harnpadoungsataya,Marketing,2008-12-12,17


## **Query 2 – Employee Count per Department**

**🟦 Proposition:**  
Show how many current employees work in each department to understand which teams are the largest.

**🟩 Why It’s Special:**  
Uses `GROUP BY` with `COUNT(*)` and filters only current employees. Efficient way to analyze staffing size department by department.

In [33]:
USE AdventureWorks2017;
GO

SELECT 
    d.Name AS DepartmentName,
    COUNT(*) AS NumberOfEmployees
FROM HumanResources.EmployeeDepartmentHistory edh
JOIN HumanResources.Department d ON edh.DepartmentID = d.DepartmentID
WHERE edh.EndDate IS NULL
GROUP BY d.Name
ORDER BY NumberOfEmployees DESC;


Commands completed successfully.

(16 rows affected)

Total execution time: 00:00:00.054

DepartmentName,NumberOfEmployees
Production,179
Sales,18
Purchasing,12
Finance,10
Information Services,10
Marketing,9
Facilities and Maintenance,7
Shipping and Receiving,6
Quality Assurance,6
Engineering,6


## **Query 3 – Most Recently Hired Employees**

**🟦 Proposition:**  
Show the 5 most recently hired employees who are still employed. Great for onboarding and performance tracking.

**🟩 Why It’s Special:**  
Uses `TOP 5` with `ORDER BY StartDate DESC` to highlight new hires. Efficient and meaningful for recent hiring insights.

In [34]:
USE AdventureWorks2017;
GO

SELECT TOP 5
    p.FirstName + ' ' + p.LastName AS EmployeeName,
    d.Name AS DepartmentName,
    edh.StartDate AS HireDate
FROM HumanResources.EmployeeDepartmentHistory edh
JOIN Person.Person p ON edh.BusinessEntityID = p.BusinessEntityID
JOIN HumanResources.Department d ON edh.DepartmentID = d.DepartmentID
WHERE edh.EndDate IS NULL
ORDER BY edh.StartDate DESC;


Commands completed successfully.

(5 rows affected)

Total execution time: 00:00:00.056

EmployeeName,DepartmentName,HireDate
Laura Norman,Executive,2013-11-14
Lynn Tsoflias,Sales,2013-05-30
Rachel Valdez,Sales,2013-05-30
Syed Abbas,Sales,2013-03-14
Tete Mensa-Annan,Sales,2012-09-30


## **Query 4 – Average Hourly Pay by Department**

**🟦 Proposition:**  
Calculate average hourly pay in each department based on employees’ most recent pay rate.

**🟩 Why It’s Special:**  
Uses a CTE with `ROW_NUMBER()` to get the **latest rate** per employee, then `AVG()` with `GROUP BY`. It’s analytical and performance-conscious.

In [35]:
USE AdventureWorks2017;
GO

WITH LatestPay AS (
    SELECT
        eph.BusinessEntityID,
        eph.Rate,
        ROW_NUMBER() OVER (PARTITION BY eph.BusinessEntityID ORDER BY eph.RateChangeDate DESC) AS rn
    FROM HumanResources.EmployeePayHistory eph
)

SELECT 
    d.Name AS DepartmentName,
    AVG(lp.Rate) AS AverageHourlyRate
FROM LatestPay lp
JOIN HumanResources.EmployeeDepartmentHistory edh 
    ON lp.BusinessEntityID = edh.BusinessEntityID
JOIN HumanResources.Department d 
    ON edh.DepartmentID = d.DepartmentID
WHERE lp.rn = 1 AND edh.EndDate IS NULL
GROUP BY d.Name
ORDER BY AverageHourlyRate DESC;


Commands completed successfully.

(16 rows affected)

Total execution time: 00:00:00.050

DepartmentName,AverageHourlyRate
Executive,92.7981
Research and Development,43.6731
Engineering,40.1442
Information Services,34.1586
Sales,29.9719
Tool Design,27.1731
Finance,23.935
Production Control,18.6794
Purchasing,18.3269
Human Resources,18.0248


## **Query 5 – Employees Without a Current Department**

**🟦 Proposition:**  
Find employees who were assigned in the past but currently have no department.

**🟩 Why It’s Special:**  
Uses `NOT EXISTS` to find employees who are **not in any current record**. Useful for spotting data issues or inactive records.

In [36]:
USE AdventureWorks2017;
GO

SELECT 
    p.FirstName + ' ' + p.LastName AS EmployeeName
FROM HumanResources.EmployeeDepartmentHistory edh
JOIN Person.Person p ON edh.BusinessEntityID = p.BusinessEntityID
WHERE edh.EndDate IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM HumanResources.EmployeeDepartmentHistory current_edh
      WHERE current_edh.BusinessEntityID = edh.BusinessEntityID
        AND current_edh.EndDate IS NULL
  )
GROUP BY p.FirstName, p.LastName;


Commands completed successfully.

(0 rows affected)

Total execution time: 00:00:00.038

EmployeeName


## **Query 6 – Top 5 Departments by Average Pay**

**🟦 Proposition:**  
Find which departments offer the highest average pay to current employees.

**🟩 Why It’s Special:**  
Builds on earlier logic using `AVG()` and `TOP 5`. Great for salary benchmarking between teams.

In [37]:
USE AdventureWorks2017;
GO

WITH LatestPay AS (
    SELECT
        eph.BusinessEntityID,
        eph.Rate,
        ROW_NUMBER() OVER (PARTITION BY eph.BusinessEntityID ORDER BY eph.RateChangeDate DESC) AS rn
    FROM HumanResources.EmployeePayHistory eph
)

SELECT TOP 5
    d.Name AS DepartmentName,
    AVG(lp.Rate) AS AverageHourlyRate
FROM LatestPay lp
JOIN HumanResources.EmployeeDepartmentHistory edh 
    ON lp.BusinessEntityID = edh.BusinessEntityID
JOIN HumanResources.Department d 
    ON edh.DepartmentID = d.DepartmentID
WHERE lp.rn = 1 AND edh.EndDate IS NULL
GROUP BY d.Name
ORDER BY AverageHourlyRate DESC;


Commands completed successfully.

(5 rows affected)

Total execution time: 00:00:00.046

DepartmentName,AverageHourlyRate
Executive,92.7981
Research and Development,43.6731
Engineering,40.1442
Information Services,34.1586
Sales,29.9719


## **Query 7 – Department Staffing and Average Tenure**

**🟦 Proposition:**  
Find how many people work in each department and what their average time at the company is.

**🟩 Why It’s Special:**  
Combines `COUNT()` and `AVG(DATEDIFF(...))` for a **2-in-1 analysis**. Shows both size and stability of departments.

In [38]:
USE AdventureWorks2017;
GO

SELECT 
    d.Name AS DepartmentName,
    COUNT(*) AS NumberOfEmployees,
    AVG(DATEDIFF(YEAR, edh.StartDate, GETDATE())) AS AvgYearsOfService
FROM HumanResources.EmployeeDepartmentHistory edh
JOIN HumanResources.Department d ON edh.DepartmentID = d.DepartmentID
WHERE edh.EndDate IS NULL
GROUP BY d.Name
ORDER BY AvgYearsOfService DESC;


Commands completed successfully.

(16 rows affected)

Total execution time: 00:00:00.050

DepartmentName,NumberOfEmployees,AvgYearsOfService
Research and Development,4,16
Production,179,16
Production Control,6,16
Human Resources,6,16
Finance,10,16
Information Services,10,16
Document Control,5,16
Quality Assurance,6,16
Engineering,6,16
Shipping and Receiving,6,16


## **Query 8 – Employees in Multiple Departments**

**🟦 Proposition:**  
Show employees who have worked in more than one department in their career.

**🟩 Why It’s Special:**  
Uses `HAVING COUNT(DISTINCT ...) > 1` to isolate diverse experience. Helps identify cross-trained or high-potential employees.

In [39]:
USE AdventureWorks2017;
GO

SELECT 
    p.FirstName + ' ' + p.LastName AS EmployeeName,
    COUNT(DISTINCT edh.DepartmentID) AS DepartmentCount
FROM HumanResources.EmployeeDepartmentHistory edh
JOIN Person.Person p ON edh.BusinessEntityID = p.BusinessEntityID
GROUP BY p.FirstName, p.LastName
HAVING COUNT(DISTINCT edh.DepartmentID) > 1
ORDER BY DepartmentCount DESC;


Commands completed successfully.

(5 rows affected)

Total execution time: 00:00:00.061

EmployeeName,DepartmentCount
Sheela Word,3
David Bradley,2
Laura Norman,2
William Vong,2
Rob Walters,2


## **Query 9 – Longest-Serving Employee per Department**

**🟦 Proposition:**  
List the current employee with the longest tenure in each department.

**🟩 Why It’s Special:**  
Uses `ROW_NUMBER()` with `PARTITION BY` to **rank employees** by tenure per department. Very powerful for team leads and legacy insights.

In [40]:
USE AdventureWorks2017;
GO

WITH Tenure AS (
    SELECT 
        edh.BusinessEntityID,
        edh.DepartmentID,
        DATEDIFF(DAY, edh.StartDate, GETDATE()) AS TenureDays,
        ROW_NUMBER() OVER (PARTITION BY edh.DepartmentID ORDER BY edh.StartDate ASC) AS rn
    FROM HumanResources.EmployeeDepartmentHistory edh
    WHERE edh.EndDate IS NULL
)

SELECT 
    d.Name AS DepartmentName,
    p.FirstName + ' ' + p.LastName AS EmployeeName,
    t.TenureDays
FROM Tenure t
JOIN HumanResources.Department d ON t.DepartmentID = d.DepartmentID
JOIN Person.Person p ON t.BusinessEntityID = p.BusinessEntityID
WHERE t.rn = 1
ORDER BY t.TenureDays DESC;


Commands completed successfully.

(16 rows affected)

Total execution time: 00:00:00.054

DepartmentName,EmployeeName,TenureDays
Production,Guy Gilbert,6859
Marketing,Kevin Brown,6649
Engineering,Roberto Tamburello,6360
Tool Design,Thierry D'Hers,6330
Production Control,Peter Krebs,5974
Information Services,Ashvini Sharma,5971
Human Resources,Paula Barreto de Mattos,5969
Shipping and Receiving,Susan Eaton,5968
Quality Assurance,Peng Wu,5966
Facilities and Maintenance,Christian Kleinerman,5961


## **Query 10 – Most Recent Hire in Each Department**

**🟦 Proposition:**  
Find the newest employee in every department to track recent hires.

**🟩 Why It’s Special:**  
Uses `ROW_NUMBER()` in reverse order. A quick way to extract **most recent hire per group** — great for HR workflows.

In [41]:
USE AdventureWorks2017;
GO

WITH RecentHires AS (
    SELECT 
        edh.BusinessEntityID,
        edh.DepartmentID,
        edh.StartDate,
        ROW_NUMBER() OVER (PARTITION BY edh.DepartmentID ORDER BY edh.StartDate DESC) AS rn
    FROM HumanResources.EmployeeDepartmentHistory edh
    WHERE edh.EndDate IS NULL
)

SELECT 
    d.Name AS DepartmentName,
    p.FirstName + ' ' + p.LastName AS EmployeeName,
    rh.StartDate AS HireDate
FROM RecentHires rh
JOIN HumanResources.Department d ON rh.DepartmentID = d.DepartmentID
JOIN Person.Person p ON rh.BusinessEntityID = p.BusinessEntityID
WHERE rh.rn = 1
ORDER BY HireDate DESC;


Commands completed successfully.

(16 rows affected)

Total execution time: 00:00:00.058

DepartmentName,EmployeeName,HireDate
Executive,Laura Norman,2013-11-14
Sales,Lynn Tsoflias,2013-05-30
Purchasing,Sheela Word,2012-07-15
Production Control,William Vong,2011-09-01
Marketing,Mary Dempsey,2011-02-14
Engineering,Sharon Salavaria,2011-01-18
Tool Design,Janice Galvin,2010-12-23
Production,Tom Vande Velde,2010-03-10
Facilities and Maintenance,Jo Berry,2010-03-07
Quality Assurance,Sootha Charncherngkha,2010-02-23
